# STIR-Net V1 — 12 staged same-sample overfit

This notebook is the first overfit experiment after the Notebook-11 corrective source patch. It assumes the local repository already contains dynamic multi-split queries, gated seeded → temporal → discovery matching, matched-positive count supervision, and the five-stage curriculum implementation.

The notebook uses an explicit notebook-local stage controller over the same model modules and **one persistent AdamW optimizer** so every transition is visible and debuggable. It does not depend on the exact public API names of the new curriculum controller.

The main goal is to determine whether sources **4, 6, and especially 9** develop genuinely distinct split masks instead of repeated copies of the same merged component. Source 9 should have `1 primary + 8 split = 9` seeded hypotheses.


In [ ]:
from pathlib import Path
from collections import defaultdict
import gc, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import _reduced_config, _repo_root, build_real_batch
from learned.stirnet.model.query_builder import QUERY_PRIMARY, QUERY_SPLIT, QUERY_TEMPORAL, QUERY_DISCOVERY
from learned.stirnet.training.checkpoint import save_checkpoint
from learned.stirnet.training.trainer import move_to_device

SEED = 40266
AMP_DTYPE = torch.float16
QUERY_NAMES = {QUERY_PRIMARY:'primary', QUERY_SPLIT:'split', QUERY_TEMPORAL:'temporal', QUERY_DISCOVERY:'discovery'}

# Conservative first staged overfit. Increase later only if the trend is still improving.
STAGE_STEPS = {
    'spatial_dense': 30,
    'temporal_dense': 20,
    'query_bootstrap': 20,
    'native_bootstrap': 10,
    'joint': 5,
}
LOG_EVERY = 5

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = REPO_ROOT/'data'/'learned'/'stirnet'/'first_overfit'/'BlastoSPIM1_F22_030_034'
RUN_DIR = REPO_ROOT/'runs'/'stirnet'/'first_overfit'/'12_staged_same_sample'
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available(): raise RuntimeError('Notebook 12 requires CUDA.')
device = torch.device('cuda')
print('Repo:', REPO_ROOT)
print('GPU:', torch.cuda.get_device_name(0))
print('Stages:', STAGE_STEPS)


## 1. Load the exact full scene and create a fresh model

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
cfg = _reduced_config()
target = batch['targets'][0]

assert sample['current_count'] == 36, sample
assert sample['target_count'] == 33, sample
assert sample['temporal_tracklets'] == 52, sample
assert 'source_ids' in target and 'source_gt_overlap' in target

print('Sample:', sample)
print('Spacing:', batch['spacing_um'][0].tolist(), 'dref:', float(batch['dref_um'][0]))
print('\nRelevant query config:')
for k,v in vars(cfg.queries).items():
    if any(x in k for x in ('split','match','radius','support','discovery')): print(' ',k,'=',v)

def prepare_device_batch(cpu_batch):
    out={}
    for k,v in cpu_batch.items():
        if k=='targets': out[k]=v
        elif k=='spatial_inputs': out[k]=v.to(device=device,dtype=AMP_DTYPE,non_blocking=True)
        elif k=='instance_labels': out[k]=v.to(device=device,dtype=torch.int32,non_blocking=True)
        else: out[k]=move_to_device(v,device)
    return out

def full_forward(model,b,return_debug=False):
    return model(
        b['spatial_inputs'],b['instance_labels'],b['spacing_um'],b['dref_um'],
        b['instance_features'],b['instance_ids'],b['instance_batch'],b['instance_centroids_um'],
        b['graph_x'],b['graph_edge_index'],b['graph_edge_attr'],b['tracklet_id'],
        b['temporal_ref_um'],b['temporal_status'],b['hypothesis_edge_index'],b['hypothesis_edge_attr'],
        b['temporal_batch'],b.get('spatial_padding_mask'),return_debug=return_debug,
    )

device_batch=prepare_device_batch(batch)
model=StirNet(cfg).to(device)
criterion=RefinementCriterion(cfg.losses,cfg.queries,cfg.training).to(device)


## 2. Stage-specific forwards and persistent optimizer groups

In [ ]:
def forward_spatial_dense(model,b):
    acq=model.acquisition(b['spacing_um'],b['dref_um'])
    pyr=model.encoder(b['spatial_inputs'],b['spacing_um'],acq,b.get('spatial_padding_mask'))
    e2=model.decoder.decode_to_e2(pyr.features[3],pyr,acq)
    _,d0,_=model.decoder.decode_from_e2(e2,pyr,acq)
    return model.dense_heads(d0)

def forward_temporal_dense(model,b):
    acq=model.acquisition(b['spacing_um'],b['dref_um'])
    pyr=model.encoder(b['spatial_inputs'],b['spacing_um'],acq,b.get('spatial_padding_mask'))
    temporal=model._build_temporal(
        b['graph_x'],b['graph_edge_index'],b['graph_edge_attr'],b['tracklet_id'],
        b['temporal_ref_um'],b['temporal_status'],b['hypothesis_edge_index'],b['hypothesis_edge_attr'],
        b['temporal_batch'],b['dref_um'])
    e3,temporal=model.cr1(pyr.features[3],pyr.spacings_um[3],temporal,b['dref_um'],acq,pyr.padding_masks[3] if pyr.padding_masks else None)
    e2=model.decoder.decode_to_e2(e3,pyr,acq)
    e2,temporal=model.cr2(e2,pyr.spacings_um[2],temporal,b['dref_um'],acq,pyr.padding_masks[2] if pyr.padding_masks else None)
    _,d0,_=model.decoder.decode_from_e2(e2,pyr,acq)
    return model.dense_heads(d0)

def unique_params(modules):
    out=[]; seen=set()
    for m in modules:
        for p in m.parameters():
            if id(p) not in seen: out.append(p); seen.add(id(p))
    return out

group_modules={
    'spatial':[model.acquisition,model.encoder,model.decoder],
    'dense':[model.dense_heads],
    'temporal':[model.graph_encoder,model.tracklet_pooler,model.temporal_builder,model.cr1,model.cr2],
    'query':[model.query_builder,model.query_decoder],
    'native':[model.native_mask_head],
}
param_groups={k:unique_params(v) for k,v in group_modules.items()}
ids=[id(p) for ps in param_groups.values() for p in ps]
assert len(ids)==len(set(ids))
assert set(ids)=={id(p) for p in model.parameters()}, 'Notebook optimizer groups do not cover the model exactly.'

BASE_LR=float(cfg.training.lr)
optimizer=torch.optim.AdamW([
    {'params':param_groups[n],'lr':BASE_LR,'weight_decay':float(cfg.training.weight_decay),'name':n}
    for n in ('spatial','dense','temporal','query','native')])
scaler=torch.amp.GradScaler('cuda',enabled=True,init_scale=1024.0)

STAGE_LR_SCALE={
 'spatial_dense': {'spatial':1.0,'dense':1.0,'temporal':0.0,'query':0.0,'native':0.0},
 'temporal_dense':{'spatial':1.0,'dense':1.0,'temporal':1.0,'query':0.0,'native':0.0},
 'query_bootstrap':{'spatial':0.0,'dense':0.5,'temporal':1.0,'query':1.0,'native':0.0},
 'native_bootstrap':{'spatial':0.0,'dense':0.5,'temporal':1.0,'query':1.0,'native':1.0},
 'joint':{'spatial':0.1,'dense':0.5,'temporal':1.0,'query':1.0,'native':1.0},
}

def set_stage(stage):
    print('\n===',stage,'===')
    for g in optimizer.param_groups:
        scale=STAGE_LR_SCALE[stage][g['name']]
        g['lr']=BASE_LR*scale
        for p in g['params']: p.requires_grad_(scale>0)
        print(f"{g['name']:8s} lr={g['lr']:.2e} trainable={sum(p.numel() for p in g['params'] if p.requires_grad):,}")


## 3. Stage-aware losses

In [ ]:
def dense_only_losses(dense):
    fg,heat,boundary=criterion._dense_losses(dense,device_batch['targets'])
    total=cfg.losses.foreground*fg+cfg.losses.center_heatmap*heat+cfg.losses.boundary*boundary
    return {'loss':total,'foreground':fg,'center_heatmap':heat,'boundary':boundary}

def output_dict(outputs):
    # Include all production tensor fields so the patched matcher receives the promoted initial refs.
    d={}
    for name in getattr(outputs,'__dataclass_fields__',{}):
        v=getattr(outputs,name)
        if torch.is_tensor(v): d[name]=v
    d.update(exist_logits=outputs.exist_logits,centers_cellscale=outputs.centers_cellscale,
             coarse_mask_logits=outputs.coarse_mask_logits,coarse_spacing_um=outputs.coarse_spacing_um,
             dref_um=outputs.dref_um,query_types=outputs.query_types,source_instance_ids=outputs.source_instance_ids)
    return d

def get_matches(outputs):
    final=output_dict(outputs)
    coarse_targets=criterion._coarse_targets(final,device_batch['targets'])
    matches=criterion._match(final,outputs.query_padding_mask,device_batch['targets'],coarse_targets)
    return final,coarse_targets,matches

def query_stage_losses(outputs,include_native=False):
    final,coarse_targets,matches=get_matches(outputs)
    exist=criterion._existence_loss(outputs.exist_logits,outputs.query_padding_mask,matches)
    cd,cf,center=criterion._coarse_losses(final,matches,device_batch['targets'],coarse_targets)
    fg,heat,boundary=criterion._dense_losses(outputs.dense_outputs,device_batch['targets'])
    zero=outputs.exist_logits.sum()*0
    aux_total=zero
    for aux in outputs.aux_outputs:
        aux_for={**aux,'dref_um':outputs.dref_um}
        aux_targets=criterion._coarse_targets(aux_for,device_batch['targets'])
        ae=criterion._existence_loss(aux['exist_logits'],outputs.query_padding_mask,matches)
        ad,af,ac=criterion._coarse_losses(aux_for,matches,device_batch['targets'],aux_targets)
        aux_total=aux_total+cfg.losses.aux_layer*(cfg.losses.exist*ae+cfg.losses.dice_coarse*ad+cfg.losses.focal_coarse*af+cfg.losses.center*ac)
    if include_native:
        hd,hf=criterion._native_mask_losses(outputs,device_batch['targets'],matches)
    else:
        hd,hf=zero,zero
    total=(cfg.losses.exist*exist+cfg.losses.dice_coarse*cd+cfg.losses.focal_coarse*cf+cfg.losses.center*center+
           cfg.losses.foreground*fg+cfg.losses.center_heatmap*heat+cfg.losses.boundary*boundary+aux_total)
    if include_native: total=total+cfg.losses.dice_hi*hd+cfg.losses.focal_hi*hf
    return {'loss':total,'exist':exist,'dice_coarse':cd,'focal_coarse':cf,'center':center,
            'dice_hi':hd,'focal_hi':hf,'foreground':fg,'center_heatmap':heat,'boundary':boundary,
            'aux':aux_total,'matched_count':torch.tensor(sum(len(m.pred_indices) for m in matches),device=total.device,dtype=torch.float32)}


## 4. Training step

In [ ]:
def ac(): return torch.autocast(device_type='cuda',dtype=AMP_DTYPE)

def train_one_step(stage):
    model.train(); criterion.train(); optimizer.zero_grad(set_to_none=True)
    with ac():
        if stage=='spatial_dense': losses=dense_only_losses(forward_spatial_dense(model,device_batch))
        elif stage=='temporal_dense': losses=dense_only_losses(forward_temporal_dense(model,device_batch))
        elif stage=='query_bootstrap': losses=query_stage_losses(full_forward(model,device_batch),include_native=False)
        elif stage=='native_bootstrap': losses=query_stage_losses(full_forward(model,device_batch),include_native=True)
        elif stage=='joint':
            out=full_forward(model,device_batch); losses=criterion(out,device_batch['targets'])
        else: raise KeyError(stage)
        loss=losses['loss']
    if not torch.isfinite(loss): raise FloatingPointError(f'nonfinite loss in {stage}')
    scaler.scale(loss).backward(); scaler.unscale_(optimizer)
    grads=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not grads: raise RuntimeError(f'no grads in {stage}')
    if not all(torch.isfinite(g).all() for g in grads): raise FloatingPointError(f'nonfinite grads in {stage}')
    gn=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.training.max_grad_norm)
    scaler.step(optimizer); scaler.update()
    outm={k:float(v.detach().float().cpu()) for k,v in losses.items() if torch.is_tensor(v) and v.numel()==1}
    outm['grad_norm_preclip']=float(torch.as_tensor(gn).detach().cpu()); outm['amp_scale']=float(scaler.get_scale())
    return outm


## 5. Diagnostic helpers: matching, dense geometry, merged-source specialization

In [ ]:
source_ids_meta=torch.as_tensor(target['source_ids']).cpu().long()
source_gt_overlap=torch.as_tensor(target['source_gt_overlap']).cpu()
source_row={int(s):i for i,s in enumerate(source_ids_meta.tolist())}
target_ids=torch.as_tensor(target['ids']).cpu().long()
target_centers=torch.as_tensor(target['centers_cellscale']).float().cpu()
gt_label_map=torch.as_tensor(target['label_map']).cpu().numpy().astype(np.int32,copy=False)
gt_flat=gt_label_map.reshape(-1)
current_labels=batch['instance_labels'][0].cpu().numpy().astype(np.int32,copy=False)
current_flat=current_labels.reshape(-1)
spacing_native=np.asarray(batch['spacing_um'][0].cpu(),dtype=np.float32)
dref_value=float(batch['dref_um'][0])
shape_native=current_labels.shape
extent_um=(np.asarray(shape_native,dtype=np.float32)-1)*spacing_native

def roc_auc(y,s):
    y=np.asarray(y); s=np.asarray(s); pos=y==1; neg=y==0
    if pos.sum()==0 or neg.sum()==0: return np.nan
    r=rankdata(s,method='average')
    return float((r[pos].sum()-pos.sum()*(pos.sum()+1)/2)/(pos.sum()*neg.sum()))

@torch.no_grad()
def hard_dense_metrics(logits,target_array,chunk=524288):
    flat=logits[0,0].reshape(-1); tf=torch.as_tensor(target_array).reshape(-1)
    inter=pc=tc=0; ps=ns=0.0; pn=nn=0
    for a in range(0,flat.numel(),chunk):
        b=min(a+chunk,flat.numel()); p=torch.sigmoid(flat[a:b].float()); t=tf[a:b].to(p.device,dtype=torch.bool,non_blocking=True); pr=p>0.5
        inter+=int((pr&t).sum().cpu()); pc+=int(pr.sum().cpu()); tc+=int(t.sum().cpu())
        if t.any(): ps+=float(p[t].sum().cpu()); pn+=int(t.sum().cpu())
        if (~t).any(): ns+=float(p[~t].sum().cpu()); nn+=int((~t).sum().cpu())
    return {'dice':2*inter/max(pc+tc,1),'pos_prob':ps/max(pn,1),'neg_prob':ns/max(nn,1)}

def dense_target(key):
    if key in target: return torch.as_tensor(target[key]).cpu().numpy()
    if key=='foreground': return (gt_label_map>0).astype(np.uint8)
    raise KeyError(key)

def match_semantics(outputs,matches):
    m=matches[0]; qt=outputs.query_types[0].detach().cpu().long(); sid=outputs.source_instance_ids[0].detach().cpu().long(); by=defaultdict(int); bad=one=0
    for q,t in zip(m.pred_indices.detach().cpu().tolist(),m.target_indices.detach().cpu().tolist()):
        q=int(q);t=int(t);typ=int(qt[q]);by[QUERY_NAMES[typ]]+=1
        if typ in (QUERY_PRIMARY,QUERY_SPLIT):
            row=source_row.get(int(sid[q])); bad+=int(row is None or not bool(source_gt_overlap[row,t]>0))
            if typ==QUERY_SPLIT and row is not None: one+=int(int((source_gt_overlap[row]>0).sum())<2)
    return {'matched':len(m.pred_indices),'unmatched_gt':sample['target_count']-len(m.pred_indices),
            'match_primary':by['primary'],'match_split':by['split'],'match_temporal':by['temporal'],'match_discovery':by['discovery'],
            'incompatible_seeded':bad,'one_gt_split_positive':one}

def component_near_center(mask,center_vox,min_voxels):
    cc,n=ndi.label(mask)
    if n==0:return np.zeros_like(mask,dtype=bool)
    c=np.rint(center_vox).astype(int)
    if np.all(c>=0) and np.all(c<np.asarray(mask.shape)):
        lab=int(cc[tuple(c)])
        if lab>0 and np.count_nonzero(cc==lab)>=min_voxels:return cc==lab
    best=None;bd=np.inf
    for lab,sl in enumerate(ndi.find_objects(cc),start=1):
        if sl is None:continue
        comp=cc[sl]==lab
        if int(comp.sum())<min_voxels:continue
        coords=np.argwhere(comp)+np.asarray([x.start for x in sl])[None]
        d=float(np.linalg.norm(coords-c[None],axis=1).min())
        if d<bd:bd=d;best=lab
    return cc==best if best is not None else np.zeros_like(mask,dtype=bool)

@torch.no_grad()
def render_sparse(outputs,q):
    idx=torch.tensor([int(q)],device=device,dtype=torch.long); logits=model.render_masks(outputs,[idx])[0][0]
    prob=torch.sigmoid(logits.float()).cpu().numpy().astype(np.float32,copy=False); binary=prob>cfg.inference.mask_threshold
    center=outputs.centers_cellscale[0,q].detach().float().cpu().numpy()*dref_value
    binary=component_near_center(binary,(center+0.5*extent_um)/spacing_native,cfg.inference.min_mask_voxels)
    out=np.flatnonzero(binary.reshape(-1)).astype(np.int64,copy=False)
    del logits,prob,binary; torch.cuda.empty_cache(); return out

def q_gt_dice(idx,t):
    gid=int(target_ids[t]); gc=int(np.count_nonzero(gt_flat==gid)); inter=int(np.count_nonzero(gt_flat[idx]==gid)) if len(idx) else 0
    return 2*inter/max(len(idx)+gc,1)

def source_gt_dice(sid,t):
    gid=int(target_ids[t]); sm=current_flat==sid; gm=gt_flat==gid
    return 2*np.count_nonzero(sm&gm)/max(int(sm.sum())+int(gm.sum()),1)

def pair_dice(a,b):
    return 2*len(np.intersect1d(a,b,assume_unique=True))/max(len(a)+len(b),1)

@torch.no_grad()
def specialization(snapshot,outputs,matches,source_ids=(4,6,9)):
    qt=outputs.query_types[0].detach().cpu().long(); sidq=outputs.source_instance_ids[0].detach().cpu().long(); valid=~outputs.query_padding_mask[0].detach().cpu()
    mmap={int(q):int(t) for q,t in zip(matches[0].pred_indices.detach().cpu().tolist(),matches[0].target_indices.detach().cpu().tolist())}
    rows=[]; pairs=[]
    for sid in source_ids:
        row=source_row.get(sid)
        if row is None:continue
        comp=torch.nonzero(source_gt_overlap[row]>0,as_tuple=False).flatten().tolist()
        qs=[int(q) for q in torch.nonzero(valid,as_tuple=False).flatten().tolist() if int(sidq[q])==sid and int(qt[q]) in (QUERY_PRIMARY,QUERY_SPLIT)]
        rendered={q:render_sparse(outputs,q) for q in qs}
        for q in qs:
            for t in comp:
                rows.append({'snapshot':snapshot,'source_id':sid,'query':q,'query_type':QUERY_NAMES[int(qt[q])],
                             'matched_target':mmap.get(q,-1),'gt_target':int(t),'assigned':mmap.get(q,-1)==int(t),
                             'source_dice':source_gt_dice(sid,int(t)),'query_dice':q_gt_dice(rendered[q],int(t)),'voxels':len(rendered[q])})
        for i,q1 in enumerate(qs):
            for q2 in qs[i+1:]:pairs.append({'snapshot':snapshot,'source_id':sid,'q1':q1,'q2':q2,'pair_dice':pair_dice(rendered[q1],rendered[q2])})
        del rendered; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows),pd.DataFrame(pairs)


## 6. Stage-boundary snapshot

In [ ]:
snapshot_rows=[]; spec_frames=[]; pair_frames=[]

@torch.no_grad()
def evaluate_snapshot(name):
    model.eval(); criterion.eval(); gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); t0=time.perf_counter()
    with ac(): out=full_forward(model,device_batch,return_debug=True); losses=criterion(out,device_batch['targets'])
    _,_,matches=get_matches(out); sem=match_semantics(out,matches)
    assert sem['incompatible_seeded']==0 and sem['one_gt_split_positive']==0
    qt=out.query_types[0].detach().cpu().long(); valid=~out.query_padding_mask[0].detach().cpu(); ep=torch.sigmoid(out.exist_logits[0]).detach().float().cpu()
    mset=set(int(q) for q in matches[0].pred_indices.detach().cpu().tolist()); vids=torch.nonzero(valid,as_tuple=False).flatten().tolist()
    y=np.array([int(int(q) in mset) for q in vids]); s=np.array([float(ep[q]) for q in vids]); mp=[float(ep[q]) for q in vids if int(q) in mset]; up=[float(ep[q]) for q in vids if int(q) not in mset]
    fg=hard_dense_metrics(out.dense_outputs['foreground_logits'],dense_target('foreground')); bd=hard_dense_metrics(out.dense_outputs['boundary_logits'],dense_target('boundary'))
    row={'snapshot':name,'loss':float(losses['loss'].detach().cpu()),'fg_dice':fg['dice'],'boundary_dice':bd['dice'],
         'fg_in':fg['pos_prob'],'fg_out':fg['neg_prob'],'boundary_on':bd['pos_prob'],'boundary_off':bd['neg_prob'],
         'coarse_soft_dice':1-float(losses['dice_coarse'].detach().cpu()),'native_soft_dice':1-float(losses['dice_hi'].detach().cpu()),
         'exist_matched':float(np.mean(mp)) if mp else np.nan,'exist_unmatched':float(np.mean(up)) if up else np.nan,'exist_auc':roc_auc(y,s),
         'accepted_q_0p5':int(((ep>0.5)&valid).sum()),'total_q':int(valid.sum()),'q_primary':int(((qt==QUERY_PRIMARY)&valid).sum()),
         'q_split':int(((qt==QUERY_SPLIT)&valid).sum()),'q_temporal':int(((qt==QUERY_TEMPORAL)&valid).sum()),'q_discovery':int(((qt==QUERY_DISCOVERY)&valid).sum()),
         'eval_s':time.perf_counter()-t0,'peak_cuda_gib':torch.cuda.max_memory_allocated()/1024**3,**sem}
    snapshot_rows.append(row); print('\nSNAPSHOT',name); display(pd.DataFrame([row]).T)
    sidq=out.source_instance_ids[0].detach().cpu().long(); s9=int(((sidq==9)&((qt==QUERY_PRIMARY)|(qt==QUERY_SPLIT))&valid).sum()); print('source 9 seeded:',s9); assert s9>=9
    sp,pair=specialization(name,out,matches); spec_frames.append(sp); pair_frames.append(pair)
    s9f=sp[sp.source_id==9]
    if len(s9f): display(s9f.pivot_table(index=['query','query_type','matched_target'],columns='gt_target',values='query_dice',aggfunc='first').round(3))
    del out,losses,matches; gc.collect(); torch.cuda.empty_cache(); return row


## 7. Fresh baseline

In [ ]:
baseline=evaluate_snapshot('step0_fresh')

## 8. Run all five stages and checkpoint every transition

In [ ]:
history=[]; global_step=0
for stage in ('spatial_dense','temporal_dense','query_bootstrap','native_bootstrap','joint'):
    set_stage(stage); n=STAGE_STEPS[stage]
    for local_step in range(1,n+1):
        global_step+=1; gc.collect(); torch.cuda.empty_cache(); t0=time.perf_counter(); m=train_one_step(stage)
        m.update(global_step=global_step,stage=stage,stage_step=local_step,seconds=time.perf_counter()-t0); history.append(m)
        if local_step==1 or local_step%LOG_EVERY==0 or local_step==n:
            keys=('loss','foreground','boundary','exist','dice_coarse','dice_hi','center','count','grad_norm_preclip','amp_scale','seconds')
            print(stage,local_step,'/',n,{k:round(m[k],6) for k in keys if k in m})
    path=RUN_DIR/f'checkpoint_{stage}.pt'
    save_checkpoint(path,model=model,optimizer=optimizer,scaler=scaler,step=global_step,epoch=0,
                    config=cfg.to_dict() if hasattr(cfg,'to_dict') else cfg,extra={'stage':stage,'notebook':12})
    print('saved',path); evaluate_snapshot('after_'+stage)

history_df=pd.DataFrame(history); snapshot_df=pd.DataFrame(snapshot_rows)
spec_df=pd.concat(spec_frames,ignore_index=True); pair_df=pd.concat(pair_frames,ignore_index=True)
history_df.to_csv(RUN_DIR/'training_history.csv',index=False); snapshot_df.to_csv(RUN_DIR/'stage_snapshots.csv',index=False)
spec_df.to_csv(RUN_DIR/'merged_source_specialization.csv',index=False); pair_df.to_csv(RUN_DIR/'merged_source_pair_overlap.csv',index=False)
print('done, global steps:',global_step)


## 9. Stage-transition plots and tables

In [ ]:
display(snapshot_df[['snapshot','fg_dice','boundary_dice','coarse_soft_dice','native_soft_dice','exist_matched','exist_unmatched','exist_auc','accepted_q_0p5','matched','unmatched_gt']])

fig,ax=plt.subplots(figsize=(11,5)); x=np.arange(len(snapshot_df))
for k,label in [('fg_dice','foreground'),('boundary_dice','boundary'),('coarse_soft_dice','coarse'),('native_soft_dice','native')]:ax.plot(x,snapshot_df[k],marker='o',label=label)
ax.set_xticks(x);ax.set_xticklabels(snapshot_df.snapshot,rotation=35,ha='right');ax.set_ylabel('Dice');ax.grid(alpha=.2);ax.legend();plt.tight_layout();plt.show()

fig,ax=plt.subplots(figsize=(11,5)); ax.plot(x,snapshot_df.exist_matched,marker='o',label='matched');ax.plot(x,snapshot_df.exist_unmatched,marker='o',label='unmatched');ax.axhline(.5,ls='--',label='0.50 threshold')
ax.set_xticks(x);ax.set_xticklabels(snapshot_df.snapshot,rotation=35,ha='right');ax.set_ylabel('existence probability');ax.grid(alpha=.2);ax.legend();plt.tight_layout();plt.show()


## 10. True merge-correction and split-slot specialization

In [ ]:
assigned=spec_df[spec_df.assigned].copy(); assigned['improvement']=assigned.query_dice-assigned.source_dice
print('Assigned query improvement over current source:')
display(assigned.groupby(['snapshot','source_id']).agg(n=('query','count'),source_dice=('source_dice','mean'),query_dice=('query_dice','mean'),improvement=('improvement','mean'),improved=('improvement',lambda x:int((x>0).sum()))).reset_index())
print('Pairwise overlap among seeded masks:')
display(pair_df.groupby(['snapshot','source_id']).agg(mean_pair=('pair_dice','mean'),median_pair=('pair_dice','median'),max_pair=('pair_dice','max')).reset_index())

print('\nSOURCE 9 PROGRESSION')
s9=spec_df[spec_df.source_id==9]
for snap in s9.snapshot.drop_duplicates():
    print('\n',snap); f=s9[s9.snapshot==snap]
    display(f.pivot_table(index=['query','query_type','matched_target'],columns='gt_target',values='query_dice',aggfunc='first').round(3))


## 11. Final production vs oracle-matched inference

In [ ]:
@torch.no_grad()
def assemble_labels(outputs,query_ids,use_exist_score):
    ep=torch.sigmoid(outputs.exist_logits[0]).detach().float().cpu(); labels=np.zeros(shape_native,np.int32); best=np.full(shape_native,-np.inf,np.float32); used=[]
    for lab,q in enumerate(query_ids,start=1):
        idx=torch.tensor([int(q)],device=device,dtype=torch.long); logits=model.render_masks(outputs,[idx])[0][0]; prob=torch.sigmoid(logits.float()).cpu().numpy().astype(np.float32,copy=False); binary=prob>cfg.inference.mask_threshold
        center=outputs.centers_cellscale[0,q].detach().float().cpu().numpy()*dref_value; binary=component_near_center(binary,(center+0.5*extent_um)/spacing_native,cfg.inference.min_mask_voxels)
        if int(binary.sum())>=cfg.inference.min_mask_voxels:
            score=prob*float(ep[q]) if use_exist_score else prob; upd=binary&(score>best); labels[upd]=lab; best[upd]=score[upd]; used.append(int(q))
        del logits,prob,binary;gc.collect();torch.cuda.empty_cache()
    return labels,used

def instance_eval(pred,gt):
    pids,pc=np.unique(pred[pred>0],return_counts=True); gids,gc_=np.unique(gt[gt>0],return_counts=True)
    if not len(pids):return {'pred_count':0,'gt_count':len(gids),'mean_dice':0.,'median_dice':0.,'positive_pairs':0,'foreground_dice':0.}
    pr={int(x):i for i,x in enumerate(pids)};gr={int(x):i for i,x in enumerate(gids)};inter=np.zeros((len(pids),len(gids)),np.int64);pos=(pred>0)&(gt>0)
    if pos.any():
        pairs,c=np.unique(np.stack([pred[pos],gt[pos]],1),axis=0,return_counts=True)
        for (p,g),n in zip(pairs.tolist(),c.tolist()):inter[pr[int(p)],gr[int(g)]]=n
    d=2*inter/np.maximum(pc[:,None]+gc_[None,:],1);r,c=linear_sum_assignment(1-d);v=d[r,c];pf=pred>0;gf=gt>0
    return {'pred_count':len(pids),'gt_count':len(gids),'mean_dice':float(v.mean()),'median_dice':float(np.median(v)),'positive_pairs':int((v>0).sum()),'foreground_dice':2*np.count_nonzero(pf&gf)/max(int(pf.sum())+int(gf.sum()),1)}

model.eval();criterion.eval();gc.collect();torch.cuda.empty_cache()
with torch.no_grad(),ac(): final_outputs=full_forward(model,device_batch,return_debug=True)
_,_,final_matches=get_matches(final_outputs); valid=~final_outputs.query_padding_mask[0].detach().cpu(); ep=torch.sigmoid(final_outputs.exist_logits[0]).detach().float().cpu()
production_q=torch.nonzero(valid&(ep>cfg.inference.final_exist_threshold),as_tuple=False).flatten().tolist(); oracle_q=final_matches[0].pred_indices.detach().cpu().tolist()
production_labels,production_used=assemble_labels(final_outputs,production_q,True); oracle_labels,oracle_used=assemble_labels(final_outputs,oracle_q,False)
production_eval=instance_eval(production_labels,gt_label_map);oracle_eval=instance_eval(oracle_labels,gt_label_map)
print('production:',production_eval);print('oracle matched:',oracle_eval)
np.save(RUN_DIR/'final_production_labels.npy',production_labels);np.save(RUN_DIR/'final_oracle_matched_labels.npy',oracle_labels)

qt=final_outputs.query_types[0].detach().cpu().long();sidq=final_outputs.source_instance_ids[0].detach().cpu().long();source9_q=[int(q) for q in torch.nonzero(valid,as_tuple=False).flatten().tolist() if int(sidq[q])==9 and int(qt[q]) in (QUERY_PRIMARY,QUERY_SPLIT)]
source9_labels,source9_used=assemble_labels(final_outputs,source9_q,False);np.save(RUN_DIR/'final_source9_seeded_labels.npy',source9_labels)
print('source9 queries:',source9_q,'nonempty:',len(source9_used))


## 12. Optional Napari inspection

In [ ]:
OPEN_NAPARI=True
if OPEN_NAPARI:
    import napari
    raw=batch['spatial_inputs'][0,0].float().cpu().numpy(); spacing=tuple(float(v) for v in batch['spacing_um'][0])
    viewer=napari.Viewer(ndisplay=3);viewer.add_image(raw,name='Raw',scale=spacing);viewer.add_labels(current_labels,name='Current CC input',scale=spacing);viewer.add_labels(gt_label_map,name='GT',scale=spacing)
    viewer.add_labels(production_labels,name='STIR-Net production',scale=spacing,opacity=.75);viewer.add_labels(oracle_labels,name='Oracle matched - ignore existence',scale=spacing,opacity=.75,visible=False)
    viewer.add_labels(source9_labels,name='Source 9 seeded masks',scale=spacing,opacity=.75,visible=False);napari.run()
else:print('Set OPEN_NAPARI=True to inspect the final 3D result.')


## 13. Save a compact final summary

In [ ]:
final_snap=snapshot_df.iloc[-1].to_dict(); s9a=assigned[(assigned.snapshot=='after_joint')&(assigned.source_id==9)]; s9p=pair_df[(pair_df.snapshot=='after_joint')&(pair_df.source_id==9)]
summary={'stage_steps':STAGE_STEPS,'total_steps':global_step,'final_snapshot':final_snap,'production_eval':production_eval,'oracle_eval':oracle_eval,
         'source9_seeded_queries':len(source9_q),'source9_nonempty_masks':len(source9_used),
         'source9_assigned_mean_dice':float(s9a.query_dice.mean()) if len(s9a) else np.nan,
         'source9_mean_improvement':float(s9a.improvement.mean()) if len(s9a) else np.nan,
         'source9_mean_pair_dice':float(s9p.pair_dice.mean()) if len(s9p) else np.nan,
         'final_exist_gap':float(final_snap['exist_matched']-final_snap['exist_unmatched'])}
print(json.dumps(summary,indent=2,default=float))
with open(RUN_DIR/'summary.json','w',encoding='utf-8') as f:json.dump(summary,f,indent=2,default=float)


# Interpretation checklist

- **After `spatial_dense`**: foreground/boundary Dice should clearly improve. If not, stop and debug Stage 1.
- **After `temporal_dense`**: dense geometry should remain strong. A sharp collapse implicates the CR transition.
- **After `query_bootstrap`**: matched existence should rise, unmatched existence should fall, coarse Dice should improve, and count/native pressure is still absent.
- **After `native_bootstrap`**: native Dice and true merged-source query Dice should improve while dense/coarse geometry stays stable.
- **After `joint`**: compare existence separation directly against `after_native_bootstrap`. A sharp drop when count is enabled means count calibration still needs work.

For **source 9**, the strongest success signal is that nine seeded masks become mostly non-empty, different rows in the query→GT Dice matrix prefer different compatible GTs, pairwise mask Dice decreases, and assigned query→GT Dice exceeds the original merged-source→GT Dice.
